# Laboratorio 3
---

**Grupo**: 8

**Miembros del Grupo**: 

* Bedón, Bruno
* Galve Mateo, Carlos (Lider)
* Sanchez Mejía, Carlos
* Giron Rene, Omar Andony
* Avalos, Wilder
* Carrasco Baltazar, Jafet Abraham

In [1]:
# Librerías
import json
import os

import pandas as pd
from agent_framework.openai import OpenAIChatClient, OpenAIEmbeddingClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

from dsc.agents.sdp_cef import get_sdp_cef_agent
from dsc.constants.common import PROJECT_ROOT
from dsc.evaluation.agent import pre_evaluar_con_rag
from dsc.evaluation.chunking import evaluar_chunks
from dsc.rag.create import coleccion_chroma, config_rag
from dsc.rag.mostrar import mostrar_fragmentos
from dsc.rag.search import preguntar_con_rag, recuperar_fragmentos
from dsc.strategies.chunking import generar_chunks

pd.set_option('display.max_colwidth', None)

load_dotenv()
endpoint_llm = os.environ["FOUNDRY_MODEL_ENDPOINT"]
model_llm = os.environ["FOUNDRY_MODEL_DEPLOYMENT"]              # gpt-5.6-luna
model_embedding = os.environ["FOUNDRY_EMBEDDING_DEPLOYMENT"]    # text-embedding-3-small

# Identidad asignada mediante Microsoft Entra ID (sin API keys)
credential = DefaultAzureCredential()

print("Endpoint:", endpoint_llm)
print("Modelo LLM:", model_llm)
print("Modelo de embeddings:", model_embedding)

coleccion_chroma: Collection(name=soluciones_pago_te3small_estructura_v1)
Endpoint: https://afdscc01.services.ai.azure.com/openai/v1
Modelo LLM: gpt-5.6-luna
Modelo de embeddings: text-embedding-3-small


### 1. Validación laboratorio 2

Todos los integrantes del equipo ejecutaron el notebook del Laboratorio 2 sin errores: el agente `BancoCuyDeOroAsesorSdP` responde con su system prompt (alcance y restricciones), mantiene la memoria de corto plazo (`InMemoryHistoryProvider`) y aplica la compactación (`TokenBudgetComposedStrategy` con `SummarizationStrategy` y `TruncationStrategy`). En este notebook se reutiliza ese mismo agente mediante `get_sdp_cef_agent`.

### Cambios del agente del laboratorio 2 que se han aplicado en el Laboratorio 3

A la sección de restricciones de `config/system_prompt.yaml` se agrega la regla de RAG:

> Para responder preguntas sobre documentos o condiciones de las soluciones de pago, utiliza únicamente la información del contexto recuperado. Si el contexto no sustenta una respuesta, indica que no dispones de información suficiente. No inventes condiciones, cifras o procedimientos. Menciona la fuente utilizada (documento y sección).

También se ajusta `config/summarize_prompt.yaml` (prompt de la `SummarizationStrategy`): el resumen no debe copiar el bloque `CONTEXTO RECUPERADO`, porque se vuelve a recuperar en cada pregunta; solo conserva las condiciones que el agente ya comunicó al cliente y su fuente.

**Impacto del RAG en la compactación:** cada consulta ahora incluye el contexto recuperado (3 chunks, unos 1,800 caracteres), por lo que el historial crece más rápido que en el Laboratorio 2 y la compactación se activará antes. El resumen debe conservar los montos y las alternativas, pero no necesita conservar el texto de los chunks, porque se recuperan de nuevo en cada pregunta.


### 3. Preparación del conocimiento y chunking

##### A. Documentos sintéticos seleccionados

Se usan **3 documentos sintéticos** de **3 categorías diferentes**, lo que permite probar filtros por metadatos. Todos son de Banco Cuy de Oro (entidad ficticia), en contexto **Perú** y con montos en **soles (S/)**:

| id | Título | categoria | tipo_documento | fuente | Contenido |
|---|---|---|---|---|---|
| doc_001 | Catálogo de soluciones de pago | `soluciones_pago` | catalogo_producto | catalogo_soluciones_pago_banco_cuy_de_oro.pdf | Reprogramación, periodo de gracia, refinanciamiento, convenio de pago, consolidación de deudas (incluye deudas de tarjetas), pagos adelantados y cambio de fecha |
| doc_002 | Política de elegibilidad, capacidad de pago y documentación | `politicas_elegibilidad` | politica_interna | politica_elegibilidad_soluciones_pago.pdf | Segmentación por días de atraso, capacidad de pago, documentos (DNI, boletas, recibos por honorarios), restricciones (derivación de consultas operativas de tarjetas), edad y desgravamen |
| doc_003 | Procedimiento de atención, tiempos y efectos | `procedimientos` | procedimiento_operativo | procedimiento_atencion_soluciones_pago.pdf | Pasos, canales (agencia, app, Banca por Teléfono), tiempos de respuesta, efecto en la central de riesgos, costos y reclamos |

Cada documento tiene `id`, `titulo`, `contenido` (texto con secciones `##`), `categoria`, `tipo_documento` y `fuente`.

In [2]:
path_documents = PROJECT_ROOT / "data/in/lab_03/documentos_sinteticos.json"

with open(path_documents, encoding="utf-8") as f:
    documentos = json.load(f)

pd.DataFrame([
    {
        "id": d["id"],
        "titulo": d["titulo"],
        "categoria": d["categoria"],
        "tipo_documento": d["tipo_documento"],
        "fuente": d["fuente"],
        "caracteres": len(d["contenido"]),
    }
    for d in documentos
])

,id,titulo,categoria,tipo_documento,fuente,caracteres
0,doc_001,Catálogo sintético de soluciones de pago para créditos efectivos personales,soluciones_pago,catalogo_producto,catalogo_soluciones_pago_banco_cuy_de_oro.pdf,4637
1,doc_002,"Política sintética de elegibilidad, capacidad de pago y documentación para soluciones de pago",politicas_elegibilidad,politica_interna,politica_elegibilidad_soluciones_pago.pdf,2998
2,doc_003,"Procedimiento sintético de atención, tiempos de respuesta y efectos de las soluciones de pago",procedimientos,procedimiento_operativo,procedimiento_atencion_soluciones_pago.pdf,3195


##### B y C. Elegir la estrategia de chunking y Estrategia de chunking: por estructura + contexto + respaldo con solapamiento

1. **División por estructura:** cada sección `## ...` del documento es un chunk (una solución de pago, una regla de política o un paso del procedimiento).
2. **Encabezado contextual:** a cada chunk se le antepone `Documento: <título> | Sección: <sección>`. No se inventa contenido: solo se repiten el título y el nombre de sección que ya existen en el documento, para que el fragmento se entienda por separado.
3. **Respaldo con solapamiento:** si una sección supera `MAX_CARACTERES`, se subdivide en oraciones completas con un solapamiento de `SOLAPAMIENTO` caracteres.
4. El párrafo introductorio (antes de la primera sección) se guarda como chunk de `seccion = "Introducción"` porque declara el alcance (créditos efectivos personales; las tarjetas solo entran cuando se consolidan en un crédito efectivo), algo útil para las preguntas de otros temas del banco.

| Parámetro | Valor | Motivo |
|---|---|---|
| Separador | Encabezados `## ` | Cada sección es una unidad de significado completa |
| `MAX_CARACTERES` | 1.200 | Las secciones miden entre 400 y 900 caracteres; el límite solo actúa si se agregan secciones largas |
| `SOLAPAMIENTO` | 150 | Mantiene la oración frontera entre subfragmentos |

In [3]:
chunks = [chunk for doc in documentos for chunk in generar_chunks(doc)]

print("Documentos:", len(documentos))
print("Chunks generados:", len(chunks))
pd.DataFrame([
    {
        "id": c["id"],
        "categoria": c["metadata"]["categoria"],
        "seccion": c["metadata"]["seccion"],
        "caracteres": len(c["text"]),
    }
    for c in chunks
])

Documentos: 3
Chunks generados: 21


,id,categoria,seccion,caracteres
0,doc_001_chunk_1,soluciones_pago,Introducción,579
1,doc_001_chunk_2,soluciones_pago,1. Reprogramación,654
2,doc_001_chunk_3,soluciones_pago,2. Periodo de gracia,661
3,doc_001_chunk_4,soluciones_pago,3. Refinanciamiento,732
4,doc_001_chunk_5,soluciones_pago,4. Convenio de pago,718
5,doc_001_chunk_6,soluciones_pago,5. Consolidación de deudas,759
6,doc_001_chunk_7,soluciones_pago,6. Pagos adelantados: pago anticipado y adelanto de cuotas,749
7,doc_001_chunk_8,soluciones_pago,7. Cambio de fecha de pago,458
8,doc_002_chunk_1,politicas_elegibilidad,Introducción,279
9,doc_002_chunk_2,politicas_elegibilidad,1. Segmentación por días de atraso,740


In [4]:
# Inspección: los fragmentos son comprensibles por separado y conservan su documento de origen
for c in chunks[:3]:
    print("ID:", c["id"])
    print("Metadata:", c["metadata"])
    print(c["text"])
    print("-" * 100)

ID: doc_001_chunk_1
Metadata: {'fuente': 'catalogo_soluciones_pago_banco_cuy_de_oro.pdf', 'titulo': 'Catálogo sintético de soluciones de pago para créditos efectivos personales', 'seccion': 'Introducción', 'numero_chunk': 1, 'categoria': 'soluciones_pago', 'tipo_documento': 'catalogo_producto', 'documento_id': 'doc_001'}
Documento: Catálogo sintético de soluciones de pago para créditos efectivos personales | Sección: Introducción
Documento 100 % sintético elaborado para el Proyecto Integrador. Aplica exclusivamente a créditos efectivos personales en soles (S/). Las deudas de tarjetas de crédito solo se incluyen cuando el cliente las consolida dentro de un crédito efectivo personal (sección 5); las consultas operativas de tarjetas, los créditos hipotecarios, los créditos vehiculares y los créditos para negocio no forman parte de este catálogo. Las condiciones, montos y plazos son ficticios.
-------------------------------------------------------------------------------------------------

In [5]:
evaluar_chunks(chunks, documentos)

✅ IDs únicos
✅ Todos tienen encabezado de documento y sección
✅ Ningún chunk vacío (> 80 caracteres)
✅ Ningún chunk supera 1450 caracteres
✅ Metadatos solo con valores simples
✅ Todas las categorías representadas

Longitud mín/prom/máx: 275 / 608 / 792 caracteres


##### ¿Qué estrategia de chunking eligieron y por qué es adecuada para los documentos de su proyecto?

Elegimos **chunking basado en estructura** (una sección `##` = un chunk), enriquecido con un **encabezado contextual** (título del documento + nombre de sección) y con **subdivisión con solapamiento** como respaldo para secciones largas.

Es adecuada porque los documentos del copiloto son catálogos, políticas y procedimientos ya organizados en secciones autocontenidas: cada sección describe **una** solución de pago (reprogramación, periodo de gracia, refinanciamiento, convenio de pago…), **una** regla de política (segmentación por días de atraso, capacidad de pago, documentos…) o **un** aspecto del procedimiento (pasos, canales, tiempos, costos). Los clientes preguntan justamente a ese nivel ("¿cuánto dura el periodo de gracia?", "¿qué puedo hacer si tengo 45 días de atraso?"), así que la unidad de recuperación coincide con la unidad de pregunta.

El tamaño fijo habría podido partir condiciones a la mitad (por ejemplo, separar "máximo 3 meses" de "paga solo intereses y seguro de desgravamen"). La fragmentación semántica no aporta mucho aquí porque la estructura ya marca los cortes de significado, y costaría embeddings extra. El encabezado contextual resuelve el problema de independencia: una sección como "Tiempos de respuesta" no dice por sí sola que se refiere a soluciones de pago de créditos efectivos personales; con el encabezado, sí, sin agregar información inventada.

##### C. Estructura de los chunks antes de almacenarlos


| Elemento del registro | Qué contiene en nuestro proyecto | Para qué se utiliza |
|---|---|---|
| `id` | `doc_00X_chunk_N` | Identificar y actualizar el fragmento con `upsert` |
| `document` | Encabezado contextual + texto de la sección | Contexto que recibe el agente |
| `embedding` | Vector de `text-embedding-3-small` (1.536 dimensiones) | Comparación semántica con la pregunta |
| `metadata` | Ver tabla siguiente | Filtrar y mostrar procedencia |

<br></br>

| Metadato | Tipo | Uso |
|---|---|---|
| `fuente` | texto | **Contexto / procedencia**: se cita en la respuesta |
| `titulo` | texto | **Contexto**: se incluye en el contexto enviado al agente |
| `seccion` | texto | **Contexto**: permite citar "archivo + sección" |
| `numero_chunk` | número | **Trazabilidad**: orden del fragmento dentro del documento |
| `categoria` | texto | **Filtro principal** (`soluciones_pago`, `politicas_elegibilidad`, `procedimientos`) |
| `tipo_documento` | texto | **Filtro secundario** (catálogo, política, procedimiento) |
| `documento_id` | texto | **Filtro / mantenimiento**: borrar o reindexar un documento completo |

##### ¿Qué metadatos seleccionaron? ¿Cuáles usarán para filtrar y cuáles para enriquecer el contexto de las respuestas?

Seleccionamos `fuente`, `titulo`, `seccion`, `numero_chunk`, `categoria`, `tipo_documento` y `documento_id`, todos como valores simples (texto o número).

- **Para filtrar:** `categoria` es el filtro principal, porque el copiloto distingue tres intenciones del cliente: *qué soluciones existen* (`soluciones_pago`), *si califica* (`politicas_elegibilidad`) y *cómo se tramita* (`procedimientos`). `tipo_documento` y `documento_id` quedan disponibles como filtros secundarios y para mantenimiento de la colección.
- **Para enriquecer el contexto:** `fuente`, `titulo` y `seccion` se insertan en el contexto que recibe el agente y le permiten citar "archivo y sección" en cada respuesta, como exige la regla de RAG. `numero_chunk` se usa para trazabilidad en las pruebas.

### 4. Generación de embeddings con Azure OpenAI

Se usa `text-embedding-3-small` mediante `OpenAIEmbeddingClient` de Microsoft Agent Framework, con autenticación por Entra ID. El mismo cliente se usa para vectorizar los chunks y las preguntas.

##### Paso 2. Probar la generación de embeddings

In [6]:
cliente_embeddings = OpenAIEmbeddingClient(
    model=model_embedding,
    base_url=endpoint_llm,
    credential=credential,
)

prueba = await cliente_embeddings.get_embeddings(
    ["Consulta ficticia de prueba para el Proyecto Integrador."]
)
print("Dimensiones del vector:", len(prueba[0].vector))
print("Primeros valores:", [round(v, 4) for v in prueba[0].vector[:5]])

Dimensiones del vector: 1536
Primeros valores: [-0.0026, -0.0073, -0.0003, 0.0038, -0.0243]


### 5. Almacenamiento y recuperación en ChromaDB

##### A. Guardar chunks, vectores y metadatos

- `embedding_function=None`: ChromaDB no genera embeddings propios; usamos los de Azure.
- `configuration={"hnsw": {"space": "cosine"}}`: distancia del coseno (`1 - similitud`).
- Antes del `upsert` se eliminan los IDs que ya no existen en la lista `chunks` (los chunks borrados de la lista no se eliminan solos de la colección).

In [7]:
if not chunks:
    raise ValueError("Primero deben generar los chunks del proyecto.")

# Los embeddings se calculan en Azure mediante Agent Framework.
textos = [chunk["text"] for chunk in chunks]
resultado_embeddings = await cliente_embeddings.get_embeddings(textos)
vectores = [item.vector for item in resultado_embeddings]

# Limpiar registros antiguos que ya no están en la lista actual de chunks
ids_actuales = {chunk["id"] for chunk in chunks}
ids_existentes = set(coleccion_chroma.get(include=[])["ids"])
ids_obsoletos = sorted(ids_existentes - ids_actuales)
if ids_obsoletos:
    coleccion_chroma.delete(ids=ids_obsoletos)
    print("Chunks obsoletos eliminados:", ids_obsoletos)

coleccion_chroma.upsert(
    ids=[chunk["id"] for chunk in chunks],
    documents=textos,
    metadatas=[chunk["metadata"] for chunk in chunks],
    embeddings=vectores,
)

print("Colección:", config_rag["nombre_coleccion"])
print("Chunks almacenados:", coleccion_chroma.count())

Colección: soluciones_pago_te3small_estructura_v1
Chunks almacenados: 21


##### B. Recuperar información y aplicar un filtro

`recuperar_fragmentos` acepta un filtro opcional por `categoria` y un **umbral de distancia** opcional (`umbral_distancia`). El umbral queda desactivado (`None`) hasta calibrarlo con pruebas propias (celda de calibración más abajo).

In [8]:
pregunta_demo = "¿Qué opciones tiene un cliente que quiere pagar una cuota más baja?"

print("=== SIN FILTRO ===")
sin_filtro = await recuperar_fragmentos(
    pregunta_demo,
    coleccion=coleccion_chroma,
    cliente_embeddings=cliente_embeddings)
mostrar_fragmentos(sin_filtro)

print("\n=== CON FILTRO categoria='politicas_elegibilidad' ===")
con_filtro = await recuperar_fragmentos(
    pregunta_demo,
    coleccion=coleccion_chroma,
    cliente_embeddings=cliente_embeddings,
    categoria="politicas_elegibilidad")
mostrar_fragmentos(con_filtro)

print("\nTexto del primer fragmento sin filtro:\n")
print(sin_filtro[0]["text"])

=== SIN FILTRO ===
doc_002_chunk_3  | politicas_elegibilidad | politica_elegibilidad_soluciones_pago.pdf › 2. Capacidad de pago | distancia=0.4127
doc_001_chunk_7  | soluciones_pago        | catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 6. Pagos adelantados: pago anticipado y adelanto de cuotas | distancia=0.4446
doc_001_chunk_8  | soluciones_pago        | catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 7. Cambio de fecha de pago | distancia=0.4535

=== CON FILTRO categoria='politicas_elegibilidad' ===
doc_002_chunk_3  | politicas_elegibilidad | politica_elegibilidad_soluciones_pago.pdf › 2. Capacidad de pago | distancia=0.4127
doc_002_chunk_5  | politicas_elegibilidad | politica_elegibilidad_soluciones_pago.pdf › 4. Restricciones generales | distancia=0.5420
doc_002_chunk_2  | politicas_elegibilidad | politica_elegibilidad_soluciones_pago.pdf › 1. Segmentación por días de atraso | distancia=0.5563

Texto del primer fragmento sin filtro:

Documento: Política sintética de elegibilidad

**Comparación sin filtro vs. con filtro** (pregunta: *"¿Qué opciones tiene un cliente que quiere pagar una cuota más baja?"*).

| Posición | Sin filtro | Distancia | Con filtro `politicas_elegibilidad` | Distancia |
|---|---|---|---|---|
| 1 | doc_002 › 2. Capacidad de pago | 0.4127 | doc_002 › 2. Capacidad de pago | 0.4127 |
| 2 | doc_001 › 6. Pagos adelantados | 0.4446 | doc_002 › 4. Restricciones generales | 0.5420 |
| 3 | doc_001 › 7. Cambio de fecha de pago | 0.4535 | doc_002 › 1. Segmentación por días de atraso | 0.5563 |

- **Sin filtro**, la búsqueda mezcla categorías: un fragmento de la política y dos del catálogo.
- **Con filtro**, todos los fragmentos son de `politicas_elegibilidad`. *Capacidad de pago* tiene la misma distancia (0.4127) en ambas búsquedas: el filtro **no cambia la similitud** de un fragmento, solo restringe los candidatos. Por eso los puestos 2 y 3 se llenan con fragmentos menos similares (0.54 y 0.56).
- **Observación:** *1. Reprogramación*, que es la solución pensada para "reducir la cuota mensual", no entró en el top 3. Una distancia baja no garantiza que el fragmento sea el más útil; por eso las pruebas de la sección 7 revisan el contenido de la respuesta.

In [9]:
# Calibración del umbral de distancia (opcional, para la prueba de información ausente)
preguntas_en_alcance = [
    "¿Cuántos convenios de pago puede tener mi crédito?",
    "Soy independiente, ¿qué documentos debo presentar?",
    "¿Cuánto cuesta la consolidación de deudas?",
]
preguntas_fuera_de_alcance = [
    "¿Cuál es el horario de la oficina principal?",
    "¿Cómo difiero una compra de la tarjeta de crédito?",
    "¿Puedo pagar la cuota con criptomonedas?",
]

filas = []
for grupo, preguntas in [("en_alcance", preguntas_en_alcance), ("fuera_de_alcance", preguntas_fuera_de_alcance)]:
    for p in preguntas:
        top1 = (await recuperar_fragmentos(
            p, coleccion=coleccion_chroma, cliente_embeddings=cliente_embeddings, top_k=1))[0]
        filas.append({"grupo": grupo, "pregunta": p, "top1": top1["id"], "distancia_top1": round(top1["distancia"], 4)})

calibracion = pd.DataFrame(filas)
display(calibracion)
print("Máx. distancia en alcance:", calibracion.query("grupo=='en_alcance'")["distancia_top1"].max())
print("Mín. distancia fuera de alcance:", calibracion.query("grupo=='fuera_de_alcance'")["distancia_top1"].min())
print("\nSi hay separación clara entre ambos valores, un umbral intermedio es candidato; si se solapan, NO usar umbral")
print("y confiar en la regla del prompt (la recuperación no es evidencia suficiente por sí sola).")

,grupo,pregunta,top1,distancia_top1
0,en_alcance,¿Cuántos convenios de pago puede tener mi crédito?,doc_001_chunk_5,0.2624
1,en_alcance,"Soy independiente, ¿qué documentos debo presentar?",doc_002_chunk_4,0.4393
2,en_alcance,¿Cuánto cuesta la consolidación de deudas?,doc_001_chunk_6,0.2820
3,fuera_de_alcance,¿Cuál es el horario de la oficina principal?,doc_003_chunk_3,0.6125
4,fuera_de_alcance,¿Cómo difiero una compra de la tarjeta de crédito?,doc_003_chunk_2,0.4643
5,fuera_de_alcance,¿Puedo pagar la cuota con criptomonedas?,doc_001_chunk_7,0.4855


Máx. distancia en alcance: 0.4393
Mín. distancia fuera de alcance: 0.4643

Si hay separación clara entre ambos valores, un umbral intermedio es candidato; si se solapan, NO usar umbral
y confiar en la regla del prompt (la recuperación no es evidencia suficiente por sí sola).


**Conclusión de la calibración:** la distancia máxima en alcance (0.4393, *"Soy independiente, ¿qué documentos debo presentar?"*) queda apenas por debajo de la mínima fuera de alcance (0.4643, *"¿Cómo difiero una compra de la tarjeta de crédito?"*). Hay separación, pero de solo 0.025 y con 3 preguntas por grupo, así que no es confiable. Además, un umbral no resolvería los casos difíciles de la sección 7: en la prueba 5 (TEA exacta) el chunk de *Refinanciamiento* tiene distancia 0.4229 y en la prueba 6 (tarjeta) el primer chunk tiene 0.4365, ambos por debajo de cualquier umbral intermedio. **Decisión:** no usar umbral (`umbral_distancia=None`) y confiar en la regla de RAG del system prompt, que fue la que resolvió esas dos pruebas.

### 6. Integrar el RAG con el agente

Se usa **el mismo agente y la misma sesión del Laboratorio 2** (con memoria y compactación). La regla de RAG ya está incluida en `config/system_prompt.yaml`.

El flujo es: pregunta → embedding → recuperación en ChromaDB → contexto → respuesta del agente.

`preguntar_con_rag` devuelve la respuesta y los fragmentos usados, para poder registrar las pruebas de la sección 7. El parámetro `s` recibe la sesión: `sesion` para conversaciones de varios turnos o una sesión nueva para pruebas independientes.

In [10]:
client = OpenAIChatClient(
    model=model_llm,
    base_url=endpoint_llm,
    credential=credential,
)

agente = get_sdp_cef_agent(client=client)
sesion = agente.create_session()

In [11]:
resultado = await preguntar_con_rag(
    "¿Qué requisitos debo cumplir para reprogramar mi crédito?",
    agente=agente,
    coleccion=coleccion_chroma, cliente_embeddings=cliente_embeddings,
    s=sesion,)
mostrar_fragmentos(resultado["fragmentos"])
print("\n" + resultado["respuesta"])

doc_001_chunk_2  | soluciones_pago        | catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 1. Reprogramación | distancia=0.3425
doc_003_chunk_5  | procedimientos         | procedimiento_atencion_soluciones_pago.pdf › 4. Efectos en el historial crediticio | distancia=0.4391
doc_001_chunk_4  | soluciones_pago        | catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 3. Refinanciamiento | distancia=0.4571

Para reprogramar tu crédito, según la información disponible, debes cumplir con estos requisitos:

- Haber pagado al menos **6 cuotas del crédito original**.
- Tener el crédito **al día o con hasta 30 días de atraso**.
- La ampliación puede ser de hasta **24 meses**, sin que el plazo total del crédito supere **72 meses**.

La reprogramación mantiene tu crédito y la tasa pactada, pero al ampliar el plazo puede aumentar el total de intereses pagados. La aprobación está sujeta a la evaluación de la entidad.

**Fuente:** *Catálogo sintético de soluciones de pago para créditos efectivos pers

In [12]:
# Conversación de varios turnos reutilizando la misma sesión (memoria + RAG)
r_a = await preguntar_con_rag(
    "Atiendo a un cliente con 45 días de mora. ¿Qué soluciones le puedo ofrecer?",
    agente=agente,
    coleccion=coleccion_chroma, cliente_embeddings=cliente_embeddings,
    s=sesion,)
print("Turno 1:\n", r_a["respuesta"], "\n")

r_b = await preguntar_con_rag(
    "De esas dos opciones, ¿cuál se demora más en responderse?",
    agente=agente,
    coleccion=coleccion_chroma, cliente_embeddings=cliente_embeddings,
    s=sesion,)
print("Turno 2 (usa la memoria para saber cuáles son 'esas dos opciones'):\n", r_b["respuesta"])
mostrar_fragmentos(r_b["fragmentos"])

Turno 1:
 Con **45 días de mora**, puedes ofrecerle estas alternativas:

1. **Refinanciamiento:** reemplaza el crédito vigente por uno nuevo, sujeto a evaluación crediticia.  
2. **Convenio de pago:** establece un acuerdo para regularizar la deuda, también sujeto a evaluación y condiciones aplicables.

La respuesta para el **refinanciamiento** puede tomar hasta **5 días hábiles** y para el **convenio de pago**, hasta **8 días hábiles**, contados desde que la documentación esté completa.

**Fuente:** *Política sintética de elegibilidad, capacidad de pago y documentación para soluciones de pago*, sección 1. Segmentación por días de atraso; y *Procedimiento sintético de atención, tiempos de respuesta y efectos de las soluciones de pago*, sección 3. Tiempos de respuesta. 

Turno 2 (usa la memoria para saber cuáles son 'esas dos opciones'):
 El **convenio de pago** demora más en responderse: hasta **8 días hábiles**. El **refinanciamiento** tiene un plazo de respuesta de hasta **5 días hábi

### 7. Pruebas y validación del RAG

Cada prueba se ejecuta en una **sesión nueva** para que sean independientes. Las preguntas están redactadas como las haría el **cliente**, porque el agente del Laboratorio 2 atiende directamente al cliente. Para cada prueba se registra: pregunta, filtro, chunks recuperados (con fuente y distancia), respuesta y una **pre-evaluación automática** basada en la evidencia esperada. La evaluación final (correcto / parcial / incorrecto) la confirma el equipo revisando el contenido.

| # | Tipo | Pregunta | Filtro | Evidencia esperada (chunk) |
|---|---|---|---|---|
| 1 | Directa | ¿Cuántos meses puede durar como máximo el periodo de gracia? | — | 3 meses (doc_001 › 2. Periodo de gracia) |
| 2 | Reformulada | Estoy con descanso médico y necesito dejar de pagar capital por un tiempo | — | 3 meses; paga intereses y seguro de desgravamen (doc_001 › 2. Periodo de gracia) |
| 3 | Otro chunk | ¿En cuánto tiempo responden una consolidación de deudas? | — | 10 días hábiles + hasta 3 (doc_003 › 3. Tiempos de respuesta) |
| 4 | Con filtro | Tengo 45 días de atraso, ¿qué soluciones de pago tengo? | `politicas_elegibilidad` | Refinanciamiento o convenio de pago (doc_002 › 1. Segmentación por días de atraso) |
| 5 | Ausente | ¿Cuál es la TEA exacta de un refinanciamiento? | — | El documento dice que **no** hay tasa fija → debe declarar información insuficiente |
| 6 | Otro tema del banco (extra) | ¿Cómo difiero una compra de mi tarjeta de crédito a 36 cuotas? | — | No es su especialidad → no inventa condiciones y deriva al canal de tarjetas: app, Banca por Internet o (01) 000-3333 (doc_002 › 4. Restricciones generales y sección `canales` del system prompt) |
| 7 | Consolidación con tarjetas (extra) | Tengo dos tarjetas y un préstamo, ¿puedo juntarlos en una sola cuota? | — | Sí: consolidación de hasta 4 deudas, incluidas tarjetas (doc_001 › 5. Consolidación de deudas). Valida la regla del system prompt que permite orientar la consolidación de tarjetas |

In [13]:
pruebas = [
    {
        "n": 1, "tipo": "Pregunta directa",
        "pregunta": "¿Cuántos meses puede durar como máximo el periodo de gracia de mi crédito efectivo personal?",
        "categoria": None,
        "chunk_esperado": "Periodo de gracia",
        "debe_contener": ["3 meses"],
        "debe_negar": False,
    },
    {
        "n": 2, "tipo": "Pregunta reformulada",
        "pregunta": "Estoy con descanso médico y necesito dejar de pagar el capital de mi préstamo por un tiempo. ¿Por cuánto tiempo me pueden dar ese alivio y qué tendría que seguir pagando?",
        "categoria": None,
        "chunk_esperado": "Periodo de gracia",
        "debe_contener": ["3 meses", "intereses", "desgravamen"],
        "debe_negar": False,
    },
    {
        "n": 3, "tipo": "Información en otro chunk",
        "pregunta": "Si solicito una consolidación de deudas, ¿en cuánto tiempo me responden?",
        "categoria": None,
        "chunk_esperado": "Tiempos de respuesta",
        "debe_contener": ["10 días hábiles"],
        "debe_negar": False,
    },
    {
        "n": 4, "tipo": "Pregunta con filtro",
        "pregunta": "Tengo 45 días de atraso en mi préstamo personal, ¿qué soluciones de pago tengo?",
        "categoria": "politicas_elegibilidad",
        "chunk_esperado": "Segmentación por días de atraso",
        "debe_contener": ["refinanciamiento", "convenio de pago"],
        "debe_negar": False,
    },
    {
        "n": 5, "tipo": "Información ausente",
        "pregunta": "¿Cuál es la TEA exacta, en porcentaje, que me cobrarían si refinancio mi préstamo?",
        "categoria": None,
        "chunk_esperado": "Refinanciamiento",
        "debe_contener": [],
        "debe_negar": True,
    },
    {
        "n": 6, "tipo": "Otro tema del banco (extra)",
        "pregunta": "¿Cómo difiero una compra de mi tarjeta de crédito a 36 cuotas?",
        "categoria": None,
        "chunk_esperado": "Restricciones generales",
        "debe_contener": ["000-3333"],
        "debe_negar": False,
    },
    {
        "n": 7, "tipo": "Consolidación con tarjetas (extra)",
        "pregunta": "Tengo dos tarjetas de crédito y un préstamo personal, ¿puedo juntarlos en una sola cuota?",
        "categoria": None,
        "chunk_esperado": "Consolidación de deudas",
        "debe_contener": ["consolidación", "S/ 5,000"],
        "debe_negar": False,
    },
]


registro = []
for p in pruebas:
    sesion_independiente = agente.create_session()
    salida = await preguntar_con_rag(
        p["pregunta"],
        agente=agente,
        coleccion=coleccion_chroma, cliente_embeddings=cliente_embeddings,
        categoria=p["categoria"], s=sesion_independiente)
    evaluacion, justificacion = pre_evaluar_con_rag(p, salida)
    registro.append({
        "n": p["n"],
        "tipo": p["tipo"],
        "pregunta": p["pregunta"],
        "filtro": p["categoria"] or "—",
        "chunks_recuperados": "; ".join(
            f"{f['id']} [{f['metadata']['fuente']} › {f['metadata']['seccion']}] d={f['distancia']:.3f}"
            for f in salida["fragmentos"]
        ),
        "respuesta": salida["respuesta"],
        "pre_evaluacion": evaluacion,
        "justificacion": justificacion,
    })
    print(f"\n===== Prueba {p['n']} – {p['tipo']} =====")
    print("Pregunta:", p["pregunta"])
    print("Filtro:", p["categoria"] or "—")
    mostrar_fragmentos(salida["fragmentos"])
    print("Respuesta:\n", salida["respuesta"])
    print(f"Pre-evaluación: {evaluacion.upper()} – {justificacion}")


===== Prueba 1 – Pregunta directa =====
Pregunta: ¿Cuántos meses puede durar como máximo el periodo de gracia de mi crédito efectivo personal?
Filtro: —
doc_001_chunk_3  | soluciones_pago        | catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 2. Periodo de gracia | distancia=0.2676
doc_001_chunk_2  | soluciones_pago        | catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 1. Reprogramación | distancia=0.4182
doc_003_chunk_4  | procedimientos         | procedimiento_atencion_soluciones_pago.pdf › 3. Tiempos de respuesta | distancia=0.4452
Respuesta:
 El periodo de gracia puede durar como máximo **3 meses**. Durante ese tiempo pagarías únicamente los intereses del periodo y el seguro de desgravamen.

**Fuente:** *Catálogo sintético de soluciones de pago para créditos efectivos personales*, sección **2. Periodo de gracia**.
Pre-evaluación: CORRECTO – Recupera el chunk esperado y la respuesta contiene la evidencia.

===== Prueba 2 – Pregunta reformulada =====
Pregunta: Estoy con descans

In [14]:
# Tabla de registro de pruebas (exportable para el informe)
tabla_pruebas = pd.DataFrame(registro)
display(tabla_pruebas[["n", "tipo", "filtro", "chunks_recuperados", "pre_evaluacion", "justificacion"]])

,n,tipo,filtro,chunks_recuperados,pre_evaluacion,justificacion
0,1,Pregunta directa,—,doc_001_chunk_3 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 2. Periodo de gracia] d=0.268; doc_001_chunk_2 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 1. Reprogramación] d=0.418; doc_003_chunk_4 [procedimiento_atencion_soluciones_pago.pdf › 3. Tiempos de respuesta] d=0.445,correcto,Recupera el chunk esperado y la respuesta contiene la evidencia.
1,2,Pregunta reformulada,—,doc_001_chunk_3 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 2. Periodo de gracia] d=0.353; doc_001_chunk_2 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 1. Reprogramación] d=0.457; doc_003_chunk_4 [procedimiento_atencion_soluciones_pago.pdf › 3. Tiempos de respuesta] d=0.478,correcto,Recupera el chunk esperado y la respuesta contiene la evidencia.
2,3,Información en otro chunk,—,doc_001_chunk_6 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 5. Consolidación de deudas] d=0.347; doc_003_chunk_4 [procedimiento_atencion_soluciones_pago.pdf › 3. Tiempos de respuesta] d=0.378; doc_003_chunk_7 [procedimiento_atencion_soluciones_pago.pdf › 6. Escalamiento y reclamos] d=0.442,correcto,Recupera el chunk esperado y la respuesta contiene la evidencia.
3,4,Pregunta con filtro,politicas_elegibilidad,doc_002_chunk_2 [politica_elegibilidad_soluciones_pago.pdf › 1. Segmentación por días de atraso] d=0.322; doc_002_chunk_5 [politica_elegibilidad_soluciones_pago.pdf › 4. Restricciones generales] d=0.449; doc_002_chunk_3 [politica_elegibilidad_soluciones_pago.pdf › 2. Capacidad de pago] d=0.508,correcto,Recupera el chunk esperado y la respuesta contiene la evidencia.
4,5,Información ausente,—,doc_001_chunk_4 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 3. Refinanciamiento] d=0.423; doc_001_chunk_2 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 1. Reprogramación] d=0.501; doc_003_chunk_5 [procedimiento_atencion_soluciones_pago.pdf › 4. Efectos en el historial crediticio] d=0.577,correcto,Reconoce que no hay evidencia suficiente y no inventa cifras.
5,6,Otro tema del banco (extra),—,doc_001_chunk_2 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 1. Reprogramación] d=0.437; doc_001_chunk_8 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 7. Cambio de fecha de pago] d=0.438; doc_001_chunk_5 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 4. Convenio de pago] d=0.443,parcial,Recuperación ok=False; falta en la respuesta: []
6,7,Consolidación con tarjetas (extra),—,doc_001_chunk_6 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 5. Consolidación de deudas] d=0.337; doc_001_chunk_2 [catalogo_soluciones_pago_banco_cuy_de_oro.pdf › 1. Reprogramación] d=0.443; doc_002_chunk_5 [politica_elegibilidad_soluciones_pago.pdf › 4. Restricciones generales] d=0.474,correcto,Recupera el chunk esperado y la respuesta contiene la evidencia.


##### Evaluación final del equipo

Revisamos el contenido de cada respuesta, no solo la pre-evaluación automática (que es una heurística por palabras clave):

| # | Tipo | Chunk principal recuperado (distancia) | Pre-evaluación | Evaluación final | Justificación |
|---|---|---|---|---|---|
| 1 | Directa | doc_001 › 2. Periodo de gracia (0.268) | correcto | **Correcto** | Responde "3 meses", agrega que paga intereses y seguro de desgravamen, y cita el catálogo y la sección. |
| 2 | Reformulada | doc_001 › 2. Periodo de gracia (0.353) | correcto | **Correcto** | La pregunta dice "descanso médico", "dejar de pagar el capital" y "alivio", y aun así recupera el periodo de gracia en primer lugar. Responde 3 meses, intereses y desgravamen, y pide el sustento. |
| 3 | Otro chunk | doc_001 › 5. Consolidación de deudas (0.347); doc_003 › 3. Tiempos de respuesta en 2.º lugar (0.378) | correcto | **Correcto** | La respuesta está en otro documento (procedimiento), que llegó en segundo lugar. Responde 10 días hábiles más hasta 3 de pago a otras entidades y cita la sección correcta. |
| 4 | Con filtro | doc_002 › 1. Segmentación por días de atraso (0.322) | correcto | **Correcto** | Los 3 chunks son de `politicas_elegibilidad`. Responde refinanciamiento o convenio de pago y agrega las restricciones del fragmento recuperado. |
| 5 | Información ausente | doc_001 › 3. Refinanciamiento (0.423) | correcto | **Correcto** | El agente dice "El contexto no indica una TEA exacta" y explica que se define en la evaluación; no inventa un porcentaje. |
| 6 | Otro tema del banco | doc_001 › 1. Reprogramación (0.437) | parcial | **Parcial** | La respuesta es correcta: dice que el contexto no cubre tarjetas y deriva a la app, la Banca por Internet o el (01) 000-3333, que es el canal de tarjetas del system prompt. Pero no se recuperó doc_002 › 4. Restricciones generales, la derivación salió de la sección `canales` del prompt y no del RAG, y no recordó en qué temas sí puede ayudar, como pide el system prompt. |
| 7 | Consolidación con tarjetas | doc_001 › 5. Consolidación de deudas (0.337) | correcto | **Correcto** | Confirma que puede unir las dos tarjetas y el préstamo, con montos de S/ 5,000 a S/ 120,000, los documentos requeridos y que las tarjetas siguen activas. No rechazó la consulta por mencionar tarjetas. |

**Resultado:** 6 correctas y 1 parcial de 7 pruebas.

**Prueba 5:** la búsqueda recuperó *Refinanciamiento* en primer lugar porque habla de la TEA, pero ese chunk **no contiene** el valor pedido. La recuperación no fue evidencia suficiente: la respuesta correcta fue reconocer que el catálogo no fija una tasa, y así lo hizo el agente.

##### ¿En qué preguntas funcionó bien el RAG y qué dificultades observaron en la recuperación o en las respuestas?**

- **Preguntas directas y de otro documento (pruebas 1 y 3).** Con el chunking por secciones y el encabezado contextual, la sección correcta llegó en el top 2 (0.268 y 0.378) y el agente citó documento y sección.
- **Pregunta reformulada (prueba 2).** Los embeddings relacionaron "descanso médico", "dejar de pagar el capital" y "alivio" con el *Periodo de gracia* (0.353). Esa es la ventaja de la búsqueda semántica frente a la coincidencia textual. Ayudó que los documentos usen los mismos términos que el agente del Lab 2 (periodo de gracia, desgravamen, descanso médico).
- **Filtro (prueba 4).** Con `categoria="politicas_elegibilidad"`, los tres chunks salieron de la política y la respuesta fue la regla exacta de 31 a 60 días. En la sección 5 se vio que el filtro no cambia la distancia, solo limita los candidatos.
- **Información ausente (prueba 5).** La regla de RAG evitó que el agente inventara una TEA, aunque el chunk de *Refinanciamiento* era el más cercano.
- **Consolidación con tarjetas (prueba 7).** El agente distinguió entre consolidar deudas de tarjetas (sí está en su alcance) y las consultas operativas de tarjetas (prueba 6, que deriva).
- **Memoria + RAG (sección 6).** En el segundo turno, el agente entendió "esas dos opciones" por la memoria de la sesión y respondió que el convenio de pago tarda más (8 vs. 5 días hábiles).

**Dificultades observadas:**

1. **La búsqueda siempre devuelve algo (prueba 6).** Para la pregunta sobre diferir una compra de tarjeta, el top 3 fue *Reprogramación*, *Cambio de fecha* y *Convenio de pago*, que no tienen relación. La sección que indica derivar al canal de tarjetas no entró. El agente respondió bien gracias a la sección `canales` del system prompt: las instrucciones del Lab 2 y el RAG se complementan.
2. **El orden por distancia no siempre coincide con la utilidad.** En la prueba 3 el chunk con la respuesta llegó en segundo lugar, y en la comparación de la sección 5 *Reprogramación* no entró en el top 3 para "cuota más baja". Con `top_k=3` una pregunta que necesite varias secciones puede quedar incompleta; se puede evaluar subir `top_k` o aplicar **reranking**: recuperar más candidatos (por ejemplo, 8) y que `gpt-5.6-luna` los reordene según si contienen la respuesta antes de enviar los 3 mejores al agente. No se implementó en este laboratorio; queda como mejora para la Tool de búsqueda (sección 8).
3. **El umbral de distancia no es confiable.** La separación en la calibración fue de solo 0.025 y no habría filtrado las pruebas 5 y 6.
4. **Perspectiva de la pregunta.** El agente atiende al cliente. En la conversación de la sección 6 la pregunta se escribió como asesor ("Atiendo a un cliente… ¿qué le puedo ofrecer?") y el agente respondió en tercera persona ("puedes ofrecerle"). Por eso las pruebas de la sección 7 se redactaron como las haría el cliente.

### 8. Diseño de Tools y Agent as a Tool (solo planificación)

> En este laboratorio **no** se implementan Tools, MCP ni Agent as a Tool. Esta sección documenta el diseño para las próximas etapas. Es la etapa ReAct anticipada en el Laboratorio 2: el agente combinará razonamiento con acciones (simulador de cuotas, consulta de saldo) para obtener datos reales en lugar de suponerlos

##### A. Tools propuestas

| Nombre | Propósito | Entradas | Salidas | Fuente | Justificación |
|---|---|---|---|---|---|
| `buscar_conocimiento_soluciones_pago` | Recuperar fragmentos del catálogo, la política o el procedimiento. El agente la invoca cuando el cliente pregunta por condiciones, requisitos, tiempos o costos. | `consulta: str`, `categoria: str \| None` (`soluciones_pago`, `politicas_elegibilidad`, `procedimientos`), `top_k: int = 3` | Lista de `{id, texto, fuente, seccion, distancia}` | RAG (ChromaDB + `text-embedding-3-small`) | Hoy la recuperación se ejecuta antes de cada pregunta, incluso en un saludo. Como Tool, el agente decide cuándo buscar, qué categoría filtrar y si necesita una segunda búsqueda. En esta Tool se podría incorporar reranking para mejorar el orden de los fragmentos (ver dificultad 2 de la sección 7). |
| `consultar_credito_cliente` | Obtener el estado del crédito efectivo personal del cliente. Se invoca al inicio de la atención (paso 2 del procedimiento), en lugar de pedirle al cliente datos que el banco ya tiene. | `id_credito_sintetico: str` | `{saldo_capital, cuota_actual, tea, cuotas_pagadas, cuotas_pendientes, dias_atraso, num_convenios, proceso_judicial: bool}` | Base de datos sintética | Son datos transaccionales; el LLM no los conoce y no debe suponerlos. Reduce las preguntas al cliente (el Lab 2 necesita 3 a 6 interacciones para recopilar datos). |
| `identificar_soluciones_elegibles` | Aplicar las reglas de la política: segmentación por días de atraso, máximo de convenios, cuotas mínimas pagadas, edad, proceso judicial, solicitudes en evaluación. | `dias_atraso: int`, `cuotas_pagadas: int`, `num_convenios: int`, `edad: int`, `proceso_judicial: bool`, `solicitud_en_evaluacion: bool` | `{soluciones_permitidas: list[str], soluciones_descartadas: list[{solucion, motivo}]}` | Reglas codificadas a partir de la política sintética (doc_002) | Las reglas son deterministas, con umbrales exactos (30, 60 y 120 días; 2 convenios). Una función da resultados consistentes y auditables; el LLM puede equivocarse en un límite. |
| `simular_cuota` | Calcular la nueva cuota (sistema francés) para reprogramación, refinanciamiento, consolidación o periodo de gracia. | `saldo: float`, `tea: float`, `plazo_meses: int`, `meses_gracia: int = 0`, `monto_adicional: float = 0` | `{cuota_mensual, total_intereses, total_a_pagar, plazo_total}` en S/ | Cálculo financiero (Python, `Decimal`) | Los LLM no son confiables en aritmética financiera. En el Lab 2 el agente solo podía dar estimaciones referenciales ("S/ 12,400 ÷ S/ 500 ≈ 25 meses"); con la Tool la cuota sería exacta. |
| `validar_capacidad_pago` | Verificar si la cuota propuesta cumple la política: 40 % del ingreso neto (dependientes y pensionistas), 35 % (independientes) y un ingreso disponible mínimo de S/ 1,300. | `ingreso_neto: float`, `tipo_ingreso: str`, `cuotas_otras_deudas: float`, `cuota_propuesta: float` | `{cumple: bool, porcentaje_endeudamiento, ingreso_disponible, motivo}` | Reglas de doc_002 + resultado de `simular_cuota` | Reemplaza el cálculo "ingresos − gastos" que el Lab 2 hace con Chain of Thought interno por una validación exacta y trazable antes de presentar alternativas (paso 5). |
| `registrar_solicitud_solucion` *(etapa posterior, con confirmación del cliente)* | Registrar la solicitud elegida por el cliente y devolver un número de solicitud. | `id_credito_sintetico`, `solucion`, `parametros_simulacion`, `documentos_recibidos: list[str]` | `{numero_solicitud, tiempo_respuesta_dias_habiles, canal}` | Sistema sintético de solicitudes | Acción que **modifica** información: requiere confirmación explícita del cliente, validación de parámetros y registro de trazabilidad. Las entradas deben pasar antes por Content Safety y la blocklist del Lab 1. |

##### B. Evaluación de Agent as a Tool

##### ¿Necesitan un agente especializado que el agente principal pueda utilizar como herramienta?

**Por ahora, no: un único agente es suficiente.** El agente tiene un solo dominio (soluciones de pago de créditos efectivos personales), un único tipo de usuario (el cliente) y un flujo lineal (consultar crédito → identificar soluciones → simular → validar → presentar máximo 3 alternativas, como en el formato de salida del Lab 2). Las partes que requieren precisión (elegibilidad, cuotas, capacidad de pago) son **deterministas** y se resuelven mejor con Tools de función que con otro LLM, que añadiría latencia, costo y un punto más de posible error. Además, el system prompt del Lab 2 ya cubre el tono empático, las derivaciones y el formato de salida.

**Candidato para una etapa posterior:** si el proyecto crece (por ejemplo, para comparar varias alternativas con argumentos para el cliente), se evaluaría un **`AgenteComparadorAlternativas`** como herramienta del agente principal:

- **Responsabilidad:** a partir de las simulaciones ya calculadas y del contexto recuperado, redactar una comparación de máximo 3 alternativas (nueva cuota, plazo, costo total y efecto en el historial crediticio) en lenguaje claro para el cliente.
- **Recibiría:** el perfil sintético del caso, la lista de alternativas elegibles con sus simulaciones y los fragmentos RAG de efectos y costos.
- **Devolvería:** una tabla comparativa y una recomendación breve con las fuentes citadas.

Se activaría solo si las pruebas muestran que el agente principal pierde calidad al combinar cálculo, política y redacción en el mismo turno.